In [1]:
from elastica._calculus import _isnan_check
from elastica.timestepper import extend_stepper_interface
from elastica import *
from elastica._elastica_numba._rod._ribbon1D import Ribbon1D
from Cases.arm_function import(
    DampingFilterBC,
    ExponentialDampingBC,
    DampingFilterBCRingRod,)

from elastica._linalg import _batch_norm

from Cases.post_processing import (plot_video_with_surface,plot_video_activation_muscle,)

import os
from elastica._rotations import _get_rotation_matrix

from itertools import groupby

from Connections import *

In [2]:
class RibbonSimulator_withOgden(BaseSystemCollection, Constraints, MemoryBlockConnections, Forcing, CallBacks):
    pass
    
n_elem = 5
start = np.array([0.0, 0.0, 0.0])
direction = np.array([0.0, 0.0, 1.0])
normal = np.array([0.0, 1.0, 0.0])
base_length = 50
thickness = 0.1
width = 5
base_area = width*thickness
density = 1.017e-6
nu = 1e-5
E = 2.77e3
poisson_ratio = 0.34
shear_modulus = E / (poisson_ratio + 1.0)


dl = base_length / n_elem
dt = 1.1e-6

origin_force = np.array([0.0, 0.0, 0.0])
end_force = np.array([0.0, -5e-4, 0.0])
ramp_up_time = 10



Ribbon_Ogden = RibbonSimulator_withOgden()

ribbon_bollean = 1

if ribbon_bollean:
    ribbon = Ribbon1D.straight_ribbon(
        n_elem,
        start,
        direction,
        normal,
        base_length,
        thickness,
        width,
        density,
        youngs_modulus=E,
        shear_modulus=shear_modulus,
        poisson_ratio = poisson_ratio,
        nu = nu,
    )
    Ribbon_Ogden.append(ribbon)
else:
    ribbon = CosseratRod.straight_rod(
        n_elem,
        start,
        direction,
        normal,
        base_length,
        thickness,
        density,
        youngs_modulus=E,
        shear_modulus=shear_modulus,
        poisson_ratio = poisson_ratio,
        nu = nu,
    )
    Ribbon_Ogden.append(ribbon)    


Ribbon_Ogden.constrain(ribbon).using(
    DampingFilterBC,
    constrained_position_idx=(0,),
    constrained_director_idx=(0,),
    filter_order=5,  # 10,
)


Ribbon_Ogden.constrain(ribbon).using(
    OneEndFixedRod, constrained_position_idx=(0,), constrained_director_idx=(0,)
)
Ribbon_Ogden.add_forcing_to(ribbon).using(
    EndpointForces, origin_force, end_force, ramp_up_time=ramp_up_time
)

gravitational_acc = -9.80665*0
Ribbon_Ogden.add_forcing_to(ribbon).using(
    GravityForces, acc_gravity=np.array([0.0, gravitational_acc, 0.0])
)



In [3]:
class RibbonOgdenCallBack(CallBackBaseClass):
    """
    Call back function for Bean Ogeden penetration
    """

    def __init__(self, step_skip: int, callback_params: dict):
        CallBackBaseClass.__init__(self)
        self.every = step_skip
        self.callback_params = callback_params

    def make_callback(self, system, time, current_step: int):

        if current_step % self.every == 0:

            self.callback_params["time"].append(time)
            self.callback_params["step"].append(current_step)
            self.callback_params["position"].append(system.position_collection.copy())
            self.callback_params["velocity"].append(system.velocity_collection.copy())
            self.callback_params["avg_velocity"].append(
                system.compute_velocity_center_of_mass()
            )

            self.callback_params["center_of_mass"].append(
                system.compute_position_center_of_mass()
            )
            self.callback_params["curvature"].append(system.kappa.copy())

            return


pp_list = defaultdict(list)
Ribbon_Ogden.collect_diagnostics(ribbon).using(
    RibbonOgdenCallBack, step_skip=10, callback_params=pp_list
)
print("Callback function added to the simulator")

Callback function added to the simulator


In [4]:
Ribbon_Ogden.finalize()
print("System finalized")

System finalized


In [5]:
final_time = 0.2
total_steps = int(final_time / dt)
print("Total steps to take", total_steps)

timestepper = PositionVerlet()

Total steps to take 181818


In [6]:
integrate(timestepper, Ribbon_Ogden, final_time, total_steps)

positions_over_time = np.array(pp_list["position"])
if (np.isnan(positions_over_time)==False).all()==False:
    print("Simulation diverge. Try lowering time step !")

  0%|          | 0/181818 [00:00<?, ?it/s]

lengths.shape (5,)
tangents.shape (3, 5)


  0%|          | 0/181818 [00:01<?, ?it/s]


TypingError: Failed in nopython mode pipeline (step: nopython frontend)
[1m[1m[1m[1mFailed in nopython mode pipeline (step: nopython frontend)
[1m[1m[1m[1mFailed in nopython mode pipeline (step: nopython frontend)
[1m[1m[1m[1mFailed in nopython mode pipeline (step: nopython frontend)
[1m[1m[1m[1mFailed in nopython mode pipeline (step: nopython frontend)
[1m[1mNo implementation of function Function(<built-in function setitem>) found for signature:
 
 >>> setitem(array(float64, 1d, C), Tuple(Literal[int](0), int64), array(float64, 1d, C))
 
There are 16 candidate implementations:
[1m     - Of which 14 did not match due to:
     Overload of function 'setitem': File: <numerous>: Line N/A.
       With argument(s): '(array(float64, 1d, C), UniTuple(int64 x 2), array(float64, 1d, C))':[0m
[1m      No match.[0m
[1m     - Of which 1 did not match due to:
     Overload in function 'SetItemBuffer.generic': File: numba\core\typing\arraydecl.py: Line 221.
       With argument

In [ ]:
pp_list["position"]

In [7]:
from IPython.display import Video
from tqdm import tqdm


def plot_video_2D(plot_params: dict, video_name="video.mp4", margin=0.2, fps=15, plan_y_pos = None):
    from matplotlib import pyplot as plt
    import matplotlib.animation as manimation

    t = np.array(plot_params["time"])
    positions_over_time = np.array(plot_params["position"])
    total_time = int(np.around(t[..., -1], 1))
    total_frames = fps * total_time
    step = round(len(t) / total_frames)

    print("creating video -- this can take a few minutes")
    FFMpegWriter = manimation.writers["ffmpeg"]
    metadata = dict(title="Movie Test", artist="Matplotlib", comment="Movie support!")
    writer = FFMpegWriter(fps=fps, metadata=metadata)

    fig = plt.figure()
    ax = fig.add_subplot(111)
    plt.axis("equal")
    if plan_y_pos!= None:
        plt.axhline(y = plan_y_pos, color = 'r', linestyle = '--', linewidth = 1) 
    rod_lines_2d = ax.plot(
        positions_over_time[0][2], positions_over_time[0][1], linewidth=6
    )[0]
    limite = np.max(positions_over_time[0])
    ax.set_xlim([0 - margin, limite + margin])
    ax.set_ylim([-limite/2 - margin, limite/2+ margin])
    with writer.saving(fig, video_name, dpi=100):
        with plt.style.context("seaborn-v0_8-whitegrid"):
            for time in range(1, len(t)-1, step):
                rod_lines_2d.set_xdata(positions_over_time[time][2])
                rod_lines_2d.set_ydata(positions_over_time[time][1])

                writer.grab_frame()
    plt.close(fig)


filename_video = "Ogden_video.mp4"
plot_video_2D(pp_list, video_name=filename_video, margin=0.2, fps=80)

Video("Ogden_video.mp4")

ZeroDivisionError: division by zero